In [ ]:
!pip install -q rdkit xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 24.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd

url = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/tox21.csv.gz"
df = pd.read_csv(url)

TASKS = [c for c in df.columns if c not in ("smiles", "mol_id")]
print(df.shape, len(TASKS))
print(df[TASKS].notna().sum())   # label counts per task
print(df[TASKS].mean())          # positive rate per task
df.head()

(7831, 14) 12
NR-AR            7265
NR-AR-LBD        6758
NR-AhR           6549
NR-Aromatase     5821
NR-ER            6193
NR-ER-LBD        6955
NR-PPAR-gamma    6450
SR-ARE           5832
SR-ATAD5         7072
SR-HSE           6467
SR-MMP           5810
SR-p53           6774
dtype: int64
NR-AR            0.042533
NR-AR-LBD        0.035070
NR-AhR           0.117270
NR-Aromatase     0.051538
NR-ER            0.128048
NR-ER-LBD        0.050324
NR-PPAR-gamma    0.028837
SR-ARE           0.161523
SR-ATAD5         0.037330
SR-HSE           0.057523
SR-MMP           0.158003
SR-p53           0.062445
dtype: float64


,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53,mol_id,smiles
0,0.0,0.0,1.0,NaN,NaN,0.0,0.0,1.0,0.0,0.0,0.0,0.0,TOX3021,CCOc1ccc2nc(S(N)(=O)=O)sc2c1
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3020,CCN1C(=O)NC(c2ccccc2)C1=O
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN,NaN,TOX3024,CC[C@]1(O)CC[C@H]2[C@@H]3CCC4=CCCC[C@@H]4[C@H]...
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3027,CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TOX20800,CC(O)(P(=O)(O)O)P(=O)(O)O


In [ ]:
import numpy as np
from rdkit import Chem, RDLogger
from rdkit.Chem.MolStandardize import rdMolStandardize
RDLogger.DisableLog("rdApp.*")

lfc = rdMolStandardize.LargestFragmentChooser()
unc = rdMolStandardize.Uncharger()

def standardize(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    try:
        mol = rdMolStandardize.Cleanup(mol)
        mol = lfc.choose(mol)      # drop counterions/salts
        mol = unc.uncharge(mol)    # neutralize
        return Chem.MolToSmiles(mol)
    except Exception:
        return None

df["std_smiles"] = df["smiles"].map(standardize)
print("unparseable:", df["std_smiles"].isna().sum())
df = df.dropna(subset=["std_smiles"]).copy()

# collapse duplicate structures; mean of 0/1 labels flags disagreement
dedup = df.groupby("std_smiles", as_index=False)[TASKS].mean()
n_dup = len(df) - len(dedup)

for t in TASKS:
    v = dedup[t].values
    dedup[t] = np.where(np.isnan(v), np.nan, np.where(v > 0.5, 1.0, np.where(v < 0.5, 0.0, np.nan)))

dedup = dedup[dedup[TASKS].notna().any(axis=1)].reset_index(drop=True)

print("duplicates collapsed:", n_dup)
print("final compounds:", len(dedup))
print(dedup[TASKS].notna().sum())

unparseable: 8
duplicates collapsed: 234
final compounds: 7589
NR-AR            7022
NR-AR-LBD        6550
NR-AhR           6340
NR-Aromatase     5630
NR-ER            6006
NR-ER-LBD        6746
NR-PPAR-gamma    6256
SR-ARE           5660
SR-ATAD5         6853
SR-HSE           6275
SR-MMP           5633
SR-p53           6573
dtype: int64


In [ ]:
from rdkit.Chem import Descriptors, rdFingerprintGenerator

# rebuild mols, drop any that fail the standardization round-trip
mols = [Chem.MolFromSmiles(s) for s in dedup["std_smiles"]]
bad = [i for i, m in enumerate(mols) if m is None]
if bad:
    print("dropping failed round-trip:", len(bad))
    dedup = dedup.drop(index=bad).reset_index(drop=True)
    mols = [Chem.MolFromSmiles(s) for s in dedup["std_smiles"]]
assert all(m is not None for m in mols)

# Morgan / ECFP4, 2048 bits
mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
X_fp = np.array([mfpgen.GetFingerprintAsNumPy(m) for m in mols], dtype=np.float32)

# RDKit descriptors
X_desc = pd.DataFrame([Descriptors.CalcMolDescriptors(m) for m in mols])
X_desc = X_desc.replace([np.inf, -np.inf], np.nan)
X_desc = X_desc.loc[:, X_desc.isna().mean() < 0.05]      # drop mostly-missing cols
X_desc = X_desc.fillna(X_desc.median())
X_desc = X_desc.loc[:, X_desc.std() > 0]                  # drop constant cols
desc_cols = list(X_desc.columns)
X_desc = X_desc.values.astype(np.float32)

Y = dedup[TASKS].values.astype(float)
MASK = ~np.isnan(Y)
print("labels present:", MASK.sum())

labels present: 75521


/tmp/ipykernel_1150/3565805976.py:23: RuntimeWarning: overflow encountered in cast
  X_desc = X_desc.values.astype(np.float32)


In [ ]:
X_desc = pd.DataFrame([Descriptors.CalcMolDescriptors(m) for m in mols])
X_desc = X_desc.replace([np.inf, -np.inf], np.nan)

# log-scale the pathological ones, then clip
for c in ["Ipc", "AvgIpc"]:
    if c in X_desc.columns:
        X_desc[c] = np.log1p(X_desc[c].clip(lower=0))

X_desc = X_desc.loc[:, X_desc.isna().mean() < 0.05]
X_desc = X_desc.fillna(X_desc.median())
X_desc = X_desc.loc[:, X_desc.std() > 0]
X_desc = X_desc.clip(-1e30, 1e30)

desc_cols = list(X_desc.columns)
X_desc = X_desc.values.astype(np.float32)

print("desc:", X_desc.shape)
print("max:", X_desc.max(), "min:", X_desc.min())
print("any inf:", np.isinf(X_desc).any())

desc: (7587, 214)
max: 17602.145 min: -104.04
any inf: False


In [ ]:
from rdkit.Chem.Scaffolds import MurckoScaffold
from collections import defaultdict

groups = defaultdict(list)
for i, m in enumerate(mols):
    try:
        s = MurckoScaffold.MurckoScaffoldSmiles(mol=m, includeChirality=False)
    except Exception:
        s = ""
    groups[s].append(i)

sets = sorted(groups.values(), key=len, reverse=True)
n = len(mols)
cut_tr, cut_va = int(0.8 * n), int(0.9 * n)

train_idx, valid_idx, test_idx = [], [], []
for s in sets:
    if len(train_idx) + len(s) <= cut_tr:
        train_idx += s
    elif len(train_idx) + len(valid_idx) + len(s) <= cut_va:
        valid_idx += s
    else:
        test_idx += s

train_idx, valid_idx, test_idx = map(np.array, (train_idx, valid_idx, test_idx))
print("scaffolds:", len(groups))
print("sizes:", len(train_idx), len(valid_idx), len(test_idx))

for name, idx in [("train", train_idx), ("valid", valid_idx), ("test", test_idx)]:
    print(name, np.round(np.nanmean(Y[idx], axis=0), 3))

scaffolds: 2267
sizes: 6069 759 759
train [0.044 0.036 0.109 0.043 0.128 0.053 0.024 0.144 0.035 0.051 0.148 0.052]
valid [0.04  0.034 0.122 0.074 0.106 0.038 0.044 0.228 0.034 0.048 0.212 0.101]
test [0.031 0.029 0.196 0.095 0.145 0.031 0.052 0.249 0.066 0.103 0.2   0.121]


In [ ]:
print("test positives per task:")
for t, n_pos, n_lab in zip(TASKS,
                           np.nansum(Y[test_idx], axis=0),
                           MASK[test_idx].sum(axis=0)):
    print(f"  {t:16s} {int(n_pos):4d} pos / {n_lab:4d} labeled")

test positives per task:
  NR-AR              22 pos /  701 labeled
  NR-AR-LBD          18 pos /  613 labeled
  NR-AhR            120 pos /  611 labeled
  NR-Aromatase       48 pos /  504 labeled
  NR-ER              79 pos /  545 labeled
  NR-ER-LBD          20 pos /  647 labeled
  NR-PPAR-gamma      30 pos /  580 labeled
  SR-ARE            121 pos /  486 labeled
  SR-ATAD5           44 pos /  664 labeled
  SR-HSE             60 pos /  580 labeled
  SR-MMP            102 pos /  511 labeled
  SR-p53             76 pos /  628 labeled


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

FEATS = {
    "fp": X_fp,
    "desc": X_desc,
    "fp+desc": np.hstack([X_fp, X_desc]),
}

def run_baseline(make_model, X, name):
    rows = []
    for j, t in enumerate(TASKS):
        tr = train_idx[MASK[train_idx, j]]
        te = test_idx[MASK[test_idx, j]]
        ytr, yte = Y[tr, j], Y[te, j]
        if len(np.unique(yte)) < 2:
            rows.append((t, np.nan, np.nan, len(te))); continue
        m = make_model()
        m.fit(X[tr], ytr)
        p = m.predict_proba(X[te])[:, 1]
        rows.append((t, roc_auc_score(yte, p), average_precision_score(yte, p), int(yte.sum())))
    df_r = pd.DataFrame(rows, columns=["task", "roc_auc", "pr_auc", "n_pos_test"])
    print(f"\n=== {name} ===")
    print(df_r.to_string(index=False))
    print(f"mean ROC-AUC {df_r.roc_auc.mean():.3f}   mean PR-AUC {df_r.pr_auc.mean():.3f}")
    return df_r

results = {}
results["dummy"] = run_baseline(
    lambda: DummyClassifier(strategy="stratified", random_state=0), X_fp, "dummy (stratified)")

results["rf_fp"] = run_baseline(
    lambda: RandomForestClassifier(n_estimators=500, min_samples_leaf=2,
                                   class_weight="balanced_subsample",
                                   n_jobs=-1, random_state=0),
    X_fp, "RF / Morgan")


=== dummy (stratified) ===
         task  roc_auc   pr_auc  n_pos_test
        NR-AR 0.469809 0.031384          22
    NR-AR-LBD 0.479832 0.029364          18
       NR-AhR 0.498345 0.195888         120
 NR-Aromatase 0.504934 0.096230          48
        NR-ER 0.508353 0.147240          79
    NR-ER-LBD 0.488317 0.030430          20
NR-PPAR-gamma 0.504848 0.052381          30
       SR-ARE 0.494928 0.247167         121
     SR-ATAD5 0.501760 0.066500          44
       SR-HSE 0.481090 0.101709          60
       SR-MMP 0.473249 0.193342         102
       SR-p53 0.482599 0.118933          76
mean ROC-AUC 0.491   mean PR-AUC 0.109

=== RF / Morgan ===
         task  roc_auc   pr_auc  n_pos_test
        NR-AR 0.524167 0.041713          22
    NR-AR-LBD 0.651074 0.212479          18
       NR-AhR 0.815173 0.542200         120
 NR-Aromatase 0.787509 0.494811          48
        NR-ER 0.639594 0.278750          79
    NR-ER-LBD 0.605662 0.105545          20
NR-PPAR-gamma 0.795273 0.308678 

In [ ]:
from xgboost import XGBClassifier

def run_baseline2(make_model, X, name):
    rows = []
    for j, t in enumerate(TASKS):
        tr = train_idx[MASK[train_idx, j]]
        te = test_idx[MASK[test_idx, j]]
        ytr, yte = Y[tr, j], Y[te, j]
        if len(np.unique(yte)) < 2:
            rows.append((t, np.nan, np.nan, len(te))); continue
        m = make_model(ytr)
        m.fit(X[tr], ytr)
        p = m.predict_proba(X[te])[:, 1]
        rows.append((t, roc_auc_score(yte, p), average_precision_score(yte, p), int(yte.sum())))
    df_r = pd.DataFrame(rows, columns=["task", "roc_auc", "pr_auc", "n_pos_test"])
    print(f"\n=== {name} ===")
    print(df_r.to_string(index=False))
    print(f"mean ROC-AUC {df_r.roc_auc.mean():.3f}   mean PR-AUC {df_r.pr_auc.mean():.3f}")
    return df_r

def make_xgb(ytr):
    spw = (len(ytr) - ytr.sum()) / max(ytr.sum(), 1)
    return XGBClassifier(
        n_estimators=400, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.6,
        scale_pos_weight=spw, tree_method="hist",
        eval_metric="logloss", n_jobs=-1, random_state=0)

results["xgb_fp"] = run_baseline2(make_xgb, X_fp, "XGB / Morgan")
results["xgb_fpdesc"] = run_baseline2(make_xgb, FEATS["fp+desc"], "XGB / Morgan+desc")


=== XGB / Morgan ===
         task  roc_auc   pr_auc  n_pos_test
        NR-AR 0.511246 0.033912          22
    NR-AR-LBD 0.753268 0.166282          18
       NR-AhR 0.788824 0.483395         120
 NR-Aromatase 0.747533 0.441851          48
        NR-ER 0.647960 0.287161          79
    NR-ER-LBD 0.621531 0.104152          20
NR-PPAR-gamma 0.831455 0.305619          30
       SR-ARE 0.708785 0.439694         121
     SR-ATAD5 0.667669 0.127749          44
       SR-HSE 0.786394 0.348290          60
       SR-MMP 0.749065 0.429397         102
       SR-p53 0.726116 0.290398          76
mean ROC-AUC 0.712   mean PR-AUC 0.288

=== XGB / Morgan+desc ===
         task  roc_auc   pr_auc  n_pos_test
        NR-AR 0.425961 0.029056          22
    NR-AR-LBD 0.763305 0.229885          18
       NR-AhR 0.860200 0.638645         120
 NR-Aromatase 0.818348 0.530875          48
        NR-ER 0.686424 0.340368          79
    NR-ER-LBD 0.658054 0.085053          20
NR-PPAR-gamma 0.836242 0.249606 

In [ ]:
results["rf_fpdesc"] = run_baseline(
    lambda: RandomForestClassifier(n_estimators=500, min_samples_leaf=2,
                                   class_weight="balanced_subsample",
                                   n_jobs=-1, random_state=0),
    FEATS["fp+desc"], "RF / Morgan+desc")


=== RF / Morgan+desc ===
         task  roc_auc   pr_auc  n_pos_test
        NR-AR 0.556032 0.038680          22
    NR-AR-LBD 0.737162 0.283939          18
       NR-AhR 0.862610 0.629712         120
 NR-Aromatase 0.809302 0.505311          48
        NR-ER 0.731352 0.386797          79
    NR-ER-LBD 0.658293 0.085957          20
NR-PPAR-gamma 0.856424 0.335045          30
       SR-ARE 0.776226 0.545966         121
     SR-ATAD5 0.725733 0.220081          44
       SR-HSE 0.775994 0.393149          60
       SR-MMP 0.848411 0.531888         102
       SR-p53 0.772502 0.360352          76
mean ROC-AUC 0.759   mean PR-AUC 0.360


In [ ]:
results["rf_desc"] = run_baseline(
    lambda: RandomForestClassifier(n_estimators=500, min_samples_leaf=2,
                                   class_weight="balanced_subsample",
                                   n_jobs=-1, random_state=0),
    X_desc, "RF / desc only")


=== RF / desc only ===
         task  roc_auc   pr_auc  n_pos_test
        NR-AR 0.564132 0.040127          22
    NR-AR-LBD 0.754995 0.294085          18
       NR-AhR 0.862305 0.633235         120
 NR-Aromatase 0.793814 0.480815          48
        NR-ER 0.738686 0.397075          79
    NR-ER-LBD 0.671850 0.074569          20
NR-PPAR-gamma 0.839636 0.310066          30
       SR-ARE 0.781275 0.542165         121
     SR-ATAD5 0.729362 0.191178          44
       SR-HSE 0.761635 0.381508          60
       SR-MMP 0.848650 0.533367         102
       SR-p53 0.759201 0.355670          76
mean ROC-AUC 0.759   mean PR-AUC 0.353


In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def make_mlp(ytr):
    return make_pipeline(
        StandardScaler(),
        MLPClassifier(hidden_layer_sizes=(512, 128), alpha=1e-4,
                      batch_size=128, learning_rate_init=1e-3,
                      max_iter=200, early_stopping=True,
                      n_iter_no_change=10, random_state=0))

results["mlp_fpdesc"] = run_baseline2(make_mlp, FEATS["fp+desc"], "MLP / Morgan+desc")
results["mlp_desc"]   = run_baseline2(make_mlp, X_desc, "MLP / desc only")


=== MLP / Morgan+desc ===
         task  roc_auc   pr_auc  n_pos_test
        NR-AR 0.422547 0.051492          22
    NR-AR-LBD 0.669655 0.228406          18
       NR-AhR 0.742227 0.435491         120
 NR-Aromatase 0.752284 0.403955          48
        NR-ER 0.621829 0.250727          79
    NR-ER-LBD 0.534928 0.047222          20
NR-PPAR-gamma 0.752788 0.211071          30
       SR-ARE 0.725065 0.498336         121
     SR-ATAD5 0.650183 0.144802          44
       SR-HSE 0.715513 0.256485          60
       SR-MMP 0.789036 0.510746         102
       SR-p53 0.709835 0.300690          76
mean ROC-AUC 0.674   mean PR-AUC 0.278

=== MLP / desc only ===
         task  roc_auc   pr_auc  n_pos_test
        NR-AR 0.477574 0.038894          22
    NR-AR-LBD 0.674790 0.219506          18
       NR-AhR 0.798133 0.581512         120
 NR-Aromatase 0.798794 0.472976          48
        NR-ER 0.615500 0.265636          79
    NR-ER-LBD 0.546093 0.072102          20
NR-PPAR-gamma 0.725939 0.1967

In [ ]:
results["xgb_desc"] = run_baseline2(make_xgb, X_desc, "XGB / desc only")


=== XGB / desc only ===
         task  roc_auc   pr_auc  n_pos_test
        NR-AR 0.495180 0.042231          22
    NR-AR-LBD 0.785247 0.235813          18
       NR-AhR 0.861405 0.648003         120
 NR-Aromatase 0.790981 0.484654          48
        NR-ER 0.697860 0.332652          79
    NR-ER-LBD 0.653828 0.092671          20
NR-PPAR-gamma 0.842848 0.277032          30
       SR-ARE 0.774890 0.567572         121
     SR-ATAD5 0.728886 0.218529          44
       SR-HSE 0.778590 0.402272          60
       SR-MMP 0.862266 0.547893         102
       SR-p53 0.773432 0.370474          76
mean ROC-AUC 0.754   mean PR-AUC 0.352


In [ ]:
j = TASKS.index("NR-AR")
te = test_idx[MASK[test_idx, j]]
pos = te[Y[te, j] == 1]

scaf = lambda i: MurckoScaffold.MurckoScaffoldSmiles(mol=mols[i], includeChirality=False)
train_scaffolds = {scaf(i) for i in train_idx}

seen = [scaf(i) in train_scaffolds for i in pos]
print(f"NR-AR test positives: {len(pos)}, scaffold seen in train: {sum(seen)}")

# compare against a task that worked
j2 = TASKS.index("SR-MMP")
te2 = test_idx[MASK[test_idx, j2]]
pos2 = te2[Y[te2, j2] == 1]
seen2 = [scaf(i) in train_scaffolds for i in pos2]
print(f"SR-MMP test positives: {len(pos2)}, scaffold seen in train: {sum(seen2)}")

NR-AR test positives: 22, scaffold seen in train: 0
SR-MMP test positives: 102, scaffold seen in train: 0


In [ ]:
from rdkit import DataStructs

fps = [mfpgen.GetFingerprint(m) for m in mols]
train_fps = [fps[i] for i in train_idx]

def max_sim_to_train(idx):
    return [max(DataStructs.BulkTanimotoSimilarity(fps[i], train_fps)) for i in idx]

for t in ["NR-AR", "SR-MMP", "NR-AhR"]:
    j = TASKS.index(t)
    te = test_idx[MASK[test_idx, j]]
    pos = te[Y[te, j] == 1]
    neg = te[Y[te, j] == 0]
    print(f"{t:10s} pos median sim {np.median(max_sim_to_train(pos)):.3f}   "
          f"neg median sim {np.median(max_sim_to_train(neg)):.3f}")

NR-AR      pos median sim 0.405   neg median sim 0.400
SR-MMP     pos median sim 0.444   neg median sim 0.400
NR-AhR     pos median sim 0.425   neg median sim 0.400


In [ ]:
from sklearn.model_selection import GroupKFold

scaf_id = np.empty(len(mols), dtype=object)
for s, idxs in groups.items():
    for i in idxs:
        scaf_id[i] = s

gkf = GroupKFold(n_splits=5)
FOLDS = list(gkf.split(np.arange(len(mols)), groups=scaf_id))
print([len(te) for _, te in FOLDS])

[1698, 1473, 1472, 1472, 1472]


In [ ]:
def run_cv(make_model, X, name, needs_y=False):
    oof = np.full(Y.shape, np.nan)
    for tr_all, te_all in FOLDS:
        for j in range(len(TASKS)):
            tr = tr_all[MASK[tr_all, j]]
            te = te_all[MASK[te_all, j]]
            ytr = Y[tr, j]
            if len(np.unique(ytr)) < 2:
                continue
            m = make_model(ytr) if needs_y else make_model()
            m.fit(X[tr], ytr)
            oof[te, j] = m.predict_proba(X[te])[:, 1]
    rows = []
    for j, t in enumerate(TASKS):
        ok = MASK[:, j] & ~np.isnan(oof[:, j])
        y, p = Y[ok, j], oof[ok, j]
        rows.append((t, roc_auc_score(y, p), average_precision_score(y, p), int(y.sum())))
    df_r = pd.DataFrame(rows, columns=["task", "roc_auc", "pr_auc", "n_pos"])
    print(f"\n=== {name} (5-fold scaffold CV) ===")
    print(df_r.to_string(index=False))
    print(f"mean ROC-AUC {df_r.roc_auc.mean():.3f}   mean PR-AUC {df_r.pr_auc.mean():.3f}")
    return df_r

In [ ]:
X_desc = pd.DataFrame([Descriptors.CalcMolDescriptors(m) for m in mols])
X_desc = X_desc.replace([np.inf, -np.inf], np.nan)

for c in ["Ipc", "AvgIpc"]:
    if c in X_desc.columns:
        X_desc[c] = np.log1p(X_desc[c].clip(lower=0))

X_desc = X_desc.loc[:, X_desc.isna().mean() < 0.05]
X_desc = X_desc.fillna(X_desc.median())
X_desc = X_desc.loc[:, X_desc.std() > 0]
X_desc = X_desc.clip(-1e30, 1e30)

desc_cols = list(X_desc.columns)
X_desc = X_desc.values.astype(np.float32)

# rebuild the combined matrix too — it inherits the same problem
FEATS["desc"] = X_desc
FEATS["fp+desc"] = np.hstack([X_fp, X_desc])

print("desc:", X_desc.shape, "max:", X_desc.max(), "any inf:", np.isinf(X_desc).any())

desc: (7587, 214) max: 17602.145 any inf: False


In [ ]:
results["rf_desc_check"] = run_baseline(
    lambda: RandomForestClassifier(n_estimators=500, min_samples_leaf=2,
                                   class_weight="balanced_subsample",
                                   n_jobs=-1, random_state=0),
    X_desc, "RF / desc (recheck)")


=== RF / desc (recheck) ===
         task  roc_auc   pr_auc  n_pos_test
        NR-AR 0.563998 0.040077          22
    NR-AR-LBD 0.755089 0.294106          18
       NR-AhR 0.862322 0.633266         120
 NR-Aromatase 0.793905 0.480856          48
        NR-ER 0.738659 0.397245          79
    NR-ER-LBD 0.672010 0.074602          20
NR-PPAR-gamma 0.839758 0.310112          30
       SR-ARE 0.781275 0.542301         121
     SR-ATAD5 0.729362 0.191178          44
       SR-HSE 0.761635 0.381508          60
       SR-MMP 0.848626 0.533321         102
       SR-p53 0.759129 0.355637          76
mean ROC-AUC 0.759   mean PR-AUC 0.353


In [ ]:
results["xgb_desc_check"] = run_baseline2(make_xgb, X_desc, "XGB / desc (recheck)")
results["xgb_fpdesc_check"] = run_baseline2(make_xgb, FEATS["fp+desc"], "XGB / fp+desc (recheck)")


=== XGB / desc (recheck) ===
         task  roc_auc   pr_auc  n_pos_test
        NR-AR 0.495180 0.042231          22
    NR-AR-LBD 0.785247 0.235813          18
       NR-AhR 0.861405 0.648003         120
 NR-Aromatase 0.790981 0.484654          48
        NR-ER 0.697860 0.332652          79
    NR-ER-LBD 0.653828 0.092671          20
NR-PPAR-gamma 0.842848 0.277032          30
       SR-ARE 0.774890 0.567572         121
     SR-ATAD5 0.728886 0.218529          44
       SR-HSE 0.778590 0.402272          60
       SR-MMP 0.862266 0.547893         102
       SR-p53 0.773432 0.370474          76
mean ROC-AUC 0.754   mean PR-AUC 0.352

=== XGB / fp+desc (recheck) ===
         task  roc_auc   pr_auc  n_pos_test
        NR-AR 0.425961 0.029056          22
    NR-AR-LBD 0.763305 0.229885          18
       NR-AhR 0.860200 0.638645         120
 NR-Aromatase 0.818348 0.530875          48
        NR-ER 0.686424 0.340368          79
    NR-ER-LBD 0.658054 0.085053          20
NR-PPAR-gamma 0.83

In [ ]:
cv = {}
cv["rf_desc"] = run_cv(
    lambda: RandomForestClassifier(n_estimators=500, min_samples_leaf=2,
                                   class_weight="balanced_subsample",
                                   n_jobs=-1, random_state=0),
    X_desc, "RF / desc")


=== RF / desc (5-fold scaffold CV) ===
         task  roc_auc   pr_auc  n_pos
        NR-AR 0.770108 0.527743    299
    NR-AR-LBD 0.854746 0.536968    230
       NR-AhR 0.871177 0.546638    750
 NR-Aromatase 0.819403 0.296028    284
        NR-ER 0.729209 0.449661    765
    NR-ER-LBD 0.826967 0.463878    336
NR-PPAR-gamma 0.823899 0.230145    175
       SR-ARE 0.783288 0.439413    902
     SR-ATAD5 0.812036 0.184380    257
       SR-HSE 0.767583 0.290520    350
       SR-MMP 0.900460 0.670636    891
       SR-p53 0.853723 0.369589    412
mean ROC-AUC 0.818   mean PR-AUC 0.417


In [ ]:
cv["xgb_desc"]   = run_cv(make_xgb, X_desc, "XGB / desc", needs_y=True)
cv["rf_fp"]      = run_cv(lambda: RandomForestClassifier(n_estimators=500, min_samples_leaf=2,
                                                         class_weight="balanced_subsample",
                                                         n_jobs=-1, random_state=0),
                          X_fp, "RF / Morgan")
cv["rf_fpdesc"]  = run_cv(lambda: RandomForestClassifier(n_estimators=500, min_samples_leaf=2,
                                                         class_weight="balanced_subsample",
                                                         n_jobs=-1, random_state=0),
                          FEATS["fp+desc"], "RF / Morgan+desc")


=== XGB / desc (5-fold scaffold CV) ===
         task  roc_auc   pr_auc  n_pos
        NR-AR 0.784147 0.538001    299
    NR-AR-LBD 0.854884 0.548938    230
       NR-AhR 0.878423 0.566143    750
 NR-Aromatase 0.827547 0.318212    284
        NR-ER 0.738332 0.460756    765
    NR-ER-LBD 0.830503 0.455137    336
NR-PPAR-gamma 0.829671 0.204250    175
       SR-ARE 0.788808 0.473244    902
     SR-ATAD5 0.807588 0.201092    257
       SR-HSE 0.756252 0.267698    350
       SR-MMP 0.911425 0.703966    891
       SR-p53 0.855559 0.383316    412
mean ROC-AUC 0.822   mean PR-AUC 0.427

=== RF / Morgan (5-fold scaffold CV) ===
         task  roc_auc   pr_auc  n_pos
        NR-AR 0.777207 0.519799    299
    NR-AR-LBD 0.863557 0.577297    230
       NR-AhR 0.857951 0.488879    750
 NR-Aromatase 0.752661 0.242939    284
        NR-ER 0.721371 0.360484    765
    NR-ER-LBD 0.788366 0.362395    336
NR-PPAR-gamma 0.772450 0.205455    175
       SR-ARE 0.724739 0.331487    902
     SR-ATAD5 0.7941

In [ ]:
cv["xgb_fp"]     = run_cv(make_xgb, X_fp, "XGB / Morgan", needs_y=True)
cv["xgb_fpdesc"] = run_cv(make_xgb, FEATS["fp+desc"], "XGB / Morgan+desc", needs_y=True)


=== XGB / Morgan (5-fold scaffold CV) ===
         task  roc_auc   pr_auc  n_pos
        NR-AR 0.735916 0.488429    299
    NR-AR-LBD 0.846001 0.560060    230
       NR-AhR 0.846965 0.485593    750
 NR-Aromatase 0.686120 0.200875    284
        NR-ER 0.686980 0.341730    765
    NR-ER-LBD 0.742837 0.340094    336
NR-PPAR-gamma 0.742176 0.139879    175
       SR-ARE 0.717078 0.352537    902
     SR-ATAD5 0.736317 0.116728    257
       SR-HSE 0.680808 0.155227    350
       SR-MMP 0.809052 0.480097    891
       SR-p53 0.742615 0.227865    412
mean ROC-AUC 0.748   mean PR-AUC 0.324

=== XGB / Morgan+desc (5-fold scaffold CV) ===
         task  roc_auc   pr_auc  n_pos
        NR-AR 0.763805 0.526594    299
    NR-AR-LBD 0.858208 0.575149    230
       NR-AhR 0.881084 0.583345    750
 NR-Aromatase 0.838667 0.343695    284
        NR-ER 0.742653 0.461680    765
    NR-ER-LBD 0.821253 0.468328    336
NR-PPAR-gamma 0.834478 0.210348    175
       SR-ARE 0.791376 0.475798    902
     SR-ATAD

In [ ]:
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV)

class TorchMLP:
    def __init__(self, pos_weight, in_dim, seed=0,
                 hidden=(512, 128), epochs=60, bs=128, lr=1e-3, wd=1e-5):
        self.pw = pos_weight
        self.seed = seed
        self.epochs = epochs
        self.bs = bs
        self.lr = lr
        self.wd = wd
        self.hidden = hidden
        self.in_dim = in_dim
        self.scaler = StandardScaler()

    def _build(self):
        torch.manual_seed(self.seed)
        layers = []
        d = self.in_dim
        for h in self.hidden:
            layers.append(nn.Linear(d, h))
            layers.append(nn.ReLU())
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.Dropout(0.3))
            d = h
        layers.append(nn.Linear(d, 1))
        return nn.Sequential(*layers).to(DEV)

device: cpu


In [ ]:
    def fit(self, X, y):
        Xs = self.scaler.fit_transform(X).astype(np.float32)
        Xt = torch.tensor(Xs, device=DEV)
        yt = torch.tensor(y.astype(np.float32), device=DEV).unsqueeze(1)
        self.net = self._build()
        opt = torch.optim.AdamW(self.net.parameters(), lr=self.lr, weight_decay=self.wd)
        w = torch.tensor([self.pw], device=DEV, dtype=torch.float32)
        lossf = nn.BCEWithLogitsLoss(pos_weight=w)
        n = len(Xt)
        self.net.train()
        for _ in range(self.epochs):
            perm = torch.randperm(n, device=DEV)
            for k in range(0, n, self.bs):
                idx = perm[k:k + self.bs]
                if len(idx) < 2:
                    continue
                opt.zero_grad()
                loss = lossf(self.net(Xt[idx]), yt[idx])
                loss.backward()
                opt.step()
        return self

    def predict_proba(self, X):
        Xs = self.scaler.transform(X).astype(np.float32)
        self.net.eval()
        with torch.no_grad():
            logits = self.net(torch.tensor(Xs, device=DEV))
            p = torch.sigmoid(logits).cpu().numpy().ravel()
        out = np.column_stack([1 - p, p])
        return out

In [ ]:
def _fit(self, X, y):
    Xs = self.scaler.fit_transform(X).astype(np.float32)
    Xt = torch.tensor(Xs, device=DEV)
    yt = torch.tensor(y.astype(np.float32), device=DEV).unsqueeze(1)
    self.net = self._build()
    opt = torch.optim.AdamW(self.net.parameters(), lr=self.lr, weight_decay=self.wd)
    w = torch.tensor([self.pw], device=DEV, dtype=torch.float32)
    lossf = nn.BCEWithLogitsLoss(pos_weight=w)
    n = len(Xt)
    self.net.train()
    for _ in range(self.epochs):
        perm = torch.randperm(n, device=DEV)
        for k in range(0, n, self.bs):
            idx = perm[k:k + self.bs]
            if len(idx) < 2:
                continue
            opt.zero_grad()
            loss = lossf(self.net(Xt[idx]), yt[idx])
            loss.backward()
            opt.step()
    return self

def _predict_proba(self, X):
    Xs = self.scaler.transform(X).astype(np.float32)
    self.net.eval()
    with torch.no_grad():
        logits = self.net(torch.tensor(Xs, device=DEV))
        p = torch.sigmoid(logits).cpu().numpy().ravel()
    return np.column_stack([1 - p, p])

TorchMLP.fit = _fit
TorchMLP.predict_proba = _predict_proba
print([m for m in dir(TorchMLP) if not m.startswith('__')])

['_build', 'fit', 'predict_proba']


In [ ]:
def make_torch_mlp(ytr, in_dim):
    pw = (len(ytr) - ytr.sum()) / max(ytr.sum(), 1)
    return TorchMLP(pos_weight=pw, in_dim=in_dim)

cv["mlp_desc"] = run_cv(
    lambda ytr: make_torch_mlp(ytr, X_desc.shape[1]),
    X_desc, "MLP weighted / desc", needs_y=True)


=== MLP weighted / desc (5-fold scaffold CV) ===
         task  roc_auc   pr_auc  n_pos
        NR-AR 0.723124 0.422937    299
    NR-AR-LBD 0.812621 0.524336    230
       NR-AhR 0.851878 0.510349    750
 NR-Aromatase 0.764039 0.236587    284
        NR-ER 0.690866 0.375616    765
    NR-ER-LBD 0.762553 0.290421    336
NR-PPAR-gamma 0.778033 0.180758    175
       SR-ARE 0.762312 0.407236    902
     SR-ATAD5 0.759276 0.179442    257
       SR-HSE 0.698030 0.180921    350
       SR-MMP 0.860350 0.615525    891
       SR-p53 0.797035 0.285783    412
mean ROC-AUC 0.772   mean PR-AUC 0.351


In [ ]:
cv["mlp_fpdesc"] = run_cv(
    lambda ytr: make_torch_mlp(ytr, FEATS["fp+desc"].shape[1]),
    FEATS["fp+desc"], "MLP weighted / Morgan+desc", needs_y=True)


=== MLP weighted / Morgan+desc (5-fold scaffold CV) ===
         task  roc_auc   pr_auc  n_pos
        NR-AR 0.756511 0.496295    299
    NR-AR-LBD 0.819408 0.564224    230
       NR-AhR 0.827175 0.443572    750
 NR-Aromatase 0.699640 0.171650    284
        NR-ER 0.672863 0.298587    765
    NR-ER-LBD 0.725233 0.261582    336
NR-PPAR-gamma 0.715534 0.122970    175
       SR-ARE 0.719694 0.349378    902
     SR-ATAD5 0.720705 0.141468    257
       SR-HSE 0.649745 0.141202    350
       SR-MMP 0.813025 0.492950    891
       SR-p53 0.716809 0.221202    412
mean ROC-AUC 0.736   mean PR-AUC 0.309


In [ ]:
!pip install rdkit deepchem xgboost scikit-learn pandas numpy -q

import deepchem as dc
import numpy as np
import pandas as pd
import requests
import time
import re
from urllib.parse import quote
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
import xgboost as xgb
from sklearn.metrics import roc_auc_score, average_precision_score, matthews_corrcoef

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 552.4/552.4 kB 19.3 MB/s eta 0:00:00


/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING W&B installed but not logged in.  Run `wandb login` or set the WANDB_API_KEY env variable.
Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead
wandb: WARNING W&B installed but not logged in.  Run `wandb login` or set the WANDB_API_KEY env variable.


In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import roc_auc_score, average_precision_score, matthews_corrcoef

tox21_url = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/tox21.csv.gz"
df = pd.read_csv(tox21_url, compression='gzip')

tasks = ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD',
         'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']

df['mol'] = df['smiles'].apply(Chem.MolFromSmiles)
n_before = len(df)
df = df[df['mol'].notna()].reset_index(drop=True)
print(f"{n_before - len(df)} molecules dropped for invalid SMILES, {len(df)} remain")

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_mols = train_df['mol'].tolist()
test_mols = test_df['mol'].tolist()

y_train_raw = train_df[tasks].values
y_test_raw = test_df[tasks].values
w_train = (~pd.isna(y_train_raw)).astype(float)
w_test = (~pd.isna(y_test_raw)).astype(float)
y_train = np.nan_to_num(y_train_raw)
y_test = np.nan_to_num(y_test_raw)

def featurize_mol(mol, radius=2, n_bits=2048):
    morgan = np.array(AllChem.GetMorganFingerprintAsBitVect(mol, radius, n_bits))
    rdkit_desc = np.array([
        Descriptors.MolWt(mol), Descriptors.MolLogP(mol),
        Descriptors.NumHDonors(mol), Descriptors.NumHAcceptors(mol),
        Descriptors.TPSA(mol), Descriptors.NumRotatableBonds(mol),
        Descriptors.RingCount(mol),
    ])
    return np.concatenate([morgan, rdkit_desc])

X_train = np.array([featurize_mol(m) for m in train_mols])
X_test = np.array([featurize_mol(m) for m in test_mols])

models = {}
results = {}
for i, task in enumerate(tasks):
    mask = w_train[:, i] > 0
    model = xgb.XGBClassifier(n_estimators=200, max_depth=6, eval_metric='logloss')
    model.fit(X_train[mask], y_train[mask, i])
    models[task] = model

    test_mask = w_test[:, i] > 0
    preds = model.predict_proba(X_test[test_mask])[:, 1]
    y_true = y_test[test_mask, i]
    results[task] = {
        'AUROC': roc_auc_score(y_true, preds),
        'AUPRC': average_precision_score(y_true, preds),
        'MCC': matthews_corrcoef(y_true, preds > 0.5)
    }

for task in tasks:
    print(task, results[task])

[23:26:04] WARNING: not removing hydrogen atom without neighbors
[23:26:05] Explicit valence for atom # 8 Al, 6, is greater than permitted
[23:26:05] Explicit valence for atom # 3 Al, 6, is greater than permitted
[23:26:05] Explicit valence for atom # 4 Al, 6, is greater than permitted
[23:26:06] Explicit valence for atom # 4 Al, 6, is greater than permitted
[23:26:06] Explicit valence for atom # 9 Al, 6, is greater than permitted
[23:26:06] Explicit valence for atom # 5 Al, 6, is greater than permitted
[23:26:07] Explicit valence for atom # 16 Al, 6, is greater than permitted
[23:26:07] Explicit valence for atom # 20 Al, 6, is greater than permitted


8 molecules dropped for invalid SMILES, 7823 remain


Streaming output truncated to the last 5000 lines.
[23:26:18] DEPRECATION WARNING: please use MorganGenerator
[23:26:18] DEPRECATION WARNING: please use MorganGenerator
[23:26:18] DEPRECATION WARNING: please use MorganGenerator
[23:26:18] DEPRECATION WARNING: please use MorganGenerator
[23:26:18] DEPRECATION WARNING: please use MorganGenerator
[23:26:18] DEPRECATION WARNING: please use MorganGenerator
[23:26:18] DEPRECATION WARNING: please use MorganGenerator
[23:26:18] DEPRECATION WARNING: please use MorganGenerator
[23:26:18] DEPRECATION WARNING: please use MorganGenerator
[23:26:18] DEPRECATION WARNING: please use MorganGenerator
[23:26:18] DEPRECATION WARNING: please use MorganGenerator
[23:26:18] DEPRECATION WARNING: please use MorganGenerator
[23:26:18] DEPRECATION WARNING: please use MorganGenerator
[23:26:18] DEPRECATION WARNING: please use MorganGenerator
[23:26:18] DEPRECATION WARNING: please use MorganGenerator
[23:26:18] DEPRECATION WARNING: please use MorganGenerator
[23:2

NR-AR {'AUROC': np.float64(0.7741723685119912), 'AUPRC': np.float64(0.5001699177029573), 'MCC': np.float64(0.5868648503605255)}
NR-AR-LBD {'AUROC': np.float64(0.8989761534473821), 'AUPRC': np.float64(0.6915939136808193), 'MCC': np.float64(0.716402292491004)}
NR-AhR {'AUROC': np.float64(0.8594774418629769), 'AUPRC': np.float64(0.5998115574063657), 'MCC': np.float64(0.45917525779234464)}
NR-Aromatase {'AUROC': np.float64(0.8190827523681496), 'AUPRC': np.float64(0.30311341171525824), 'MCC': np.float64(0.2926423644751305)}
NR-ER {'AUROC': np.float64(0.7330278250370451), 'AUPRC': np.float64(0.47653557496212107), 'MCC': np.float64(0.4355021016982511)}
NR-ER-LBD {'AUROC': np.float64(0.8284701114488348), 'AUPRC': np.float64(0.5307381842324511), 'MCC': np.float64(0.553182611100596)}
NR-PPAR-gamma {'AUROC': np.float64(0.8576304611650484), 'AUPRC': np.float64(0.22223575196888498), 'MCC': np.float64(0.24119704770508052)}
SR-ARE {'AUROC': np.float64(0.8240172481293389), 'AUPRC': np.float64(0.552961

In [ ]:
API_KEY = os.environ.get("OPENFDA_API_KEY", "")  # from https://open.fda.gov/apis/authentication/

def fetch_faers_pediatric(limit=1000, max_records=5000):
    records = []
    skip = 0
    base_url = "https://api.fda.gov/drug/event.json"
    query = '(patient.patientonsetageunit:801) AND (patient.patientonsetage:[0 TO 17])'

    while skip < max_records:
        params = {'search': query, 'limit': limit, 'skip': skip, 'api_key': API_KEY}
        r = requests.get(base_url, params=params)

        if r.status_code == 429:
            print(f"[skip={skip}] Rate limited, waiting 60s")
            time.sleep(60)
            continue
        if r.status_code != 200:
            print(f"[skip={skip}] Status {r.status_code}: {r.text[:300]}")
            break

        data = r.json().get('results', [])
        print(f"[skip={skip}] got {len(data)} records")
        if not data:
            break
        records.extend(data)
        skip += limit
        time.sleep(0.3)

    return records

faers_raw = fetch_faers_pediatric()
print(f"Pulled {len(faers_raw)} pediatric FAERS reports")

[skip=0] got 1000 records
[skip=1000] got 1000 records
[skip=2000] got 1000 records
[skip=3000] got 1000 records
[skip=4000] got 1000 records
Pulled 5000 pediatric FAERS reports


In [ ]:
def parse_faers_records(records):
    rows = []
    for rec in records:
        serious = rec.get('serious') == '1'
        for d in rec.get('patient', {}).get('drug', []):
            name = d.get('medicinalproduct')
            if name:
                rows.append({'drug_name': name.strip().upper(), 'serious': serious})
    return pd.DataFrame(rows)

faers_df = parse_faers_records(faers_raw)
print(f"{len(faers_df)} drug mentions across {faers_df['drug_name'].nunique()} unique names")

raw_agg = faers_df.groupby('drug_name')['serious'].agg(['mean', 'count']).reset_index()
pediatric_drugs = raw_agg[raw_agg['count'] >= 5].copy()
pediatric_drugs['pediatric_toxic'] = (pediatric_drugs['mean'] >= 0.5).astype(int)
print(f"{len(pediatric_drugs)} drugs with enough reports")

15300 drug mentions across 3411 unique names
585 drugs with enough reports


In [ ]:
def clean_name(name):
    name = re.sub(r'\d+(\.\d+)?\s*(MG|MCG|G|ML|%)\b', '', name, flags=re.IGNORECASE)
    name = re.sub(r'\b(TABLET|CAPSULE|INJECTION|SOLUTION|ORAL|CHEWABLE|DR|ER|XR|HCL|AQ)\b', '', name, flags=re.IGNORECASE)
    return name.strip()

def name_to_smiles(name):
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{quote(name)}/property/IsomericSMILES,ConnectivitySMILES/JSON"
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            props = r.json()['PropertyTable']['Properties'][0]
            return props.get('IsomericSMILES') or props.get('ConnectivitySMILES')
    except Exception as e:
        print(name, "error:", e)
    return None

def name_to_smiles_with_fallback(name):
    smi = name_to_smiles(name)
    if smi:
        return smi
    cleaned = clean_name(name)
    if cleaned != name:
        return name_to_smiles(cleaned)
    return None

smiles_map = {}
unique_names = pediatric_drugs['drug_name'].unique()
for i, name in enumerate(unique_names):
    smi = name_to_smiles_with_fallback(name)
    if smi:
        smiles_map[name] = smi
    if i % 25 == 0:
        print(f"{i}/{len(unique_names)} processed, {len(smiles_map)} matched so far")
    time.sleep(0.2)

pediatric_drugs['smiles'] = pediatric_drugs['drug_name'].map(smiles_map)
mapped = pediatric_drugs.dropna(subset=['smiles']).copy()
print(f"Mapped {len(mapped)} drugs to structures")

0/585 processed, 1 matched so far
25/585 processed, 15 matched so far
50/585 processed, 33 matched so far
75/585 processed, 50 matched so far
100/585 processed, 72 matched so far
125/585 processed, 89 matched so far
150/585 processed, 106 matched so far
175/585 processed, 124 matched so far
200/585 processed, 144 matched so far
225/585 processed, 165 matched so far
250/585 processed, 179 matched so far
275/585 processed, 193 matched so far
300/585 processed, 212 matched so far
325/585 processed, 230 matched so far
350/585 processed, 249 matched so far
375/585 processed, 267 matched so far
400/585 processed, 281 matched so far
425/585 processed, 298 matched so far
450/585 processed, 318 matched so far
475/585 processed, 331 matched so far
500/585 processed, 350 matched so far
525/585 processed, 366 matched so far
550/585 processed, 388 matched so far
575/585 processed, 406 matched so far
Mapped 415 drugs to structures


In [ ]:
def canonical(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return Chem.MolToSmiles(mol) if mol else None

tox21_smiles = set()
for mol in train_mols + test_mols:
    if mol:
        tox21_smiles.add(Chem.MolToSmiles(mol))

mapped['canonical_smiles'] = mapped['smiles'].apply(canonical)
overlap = mapped.dropna(subset=['canonical_smiles']).copy()
overlap = overlap[overlap['canonical_smiles'].isin(tox21_smiles)]
print(f"{len(overlap)} pediatric-labeled drugs overlap with Tox21 adult compounds")

137 pediatric-labeled drugs overlap with Tox21 adult compounds


In [ ]:
overlap_mols = [Chem.MolFromSmiles(s) for s in overlap['canonical_smiles']]
X_pediatric = np.array([featurize_mol(m) for m in overlap_mols])
y_pediatric = overlap['pediatric_toxic'].values

endpoint_preds = np.column_stack([
    models[task].predict_proba(X_pediatric)[:, 1] for task in tasks
])
pediatric_risk_score = endpoint_preds.mean(axis=1)

pediatric_auroc = roc_auc_score(y_pediatric, pediatric_risk_score)
pediatric_auprc = average_precision_score(y_pediatric, pediatric_risk_score)
pediatric_mcc = matthews_corrcoef(y_pediatric, pediatric_risk_score > 0.5)

adult_auroc_mean = np.mean([results[task]['AUROC'] for task in tasks])
transfer_gap = adult_auroc_mean - pediatric_auroc

print(f"Compounds evaluated: {len(y_pediatric)}")
print(f"Adult AUROC (mean across endpoints): {adult_auroc_mean:.3f}")
print(f"Pediatric AUROC: {pediatric_auroc:.3f}")
print(f"Pediatric AUPRC: {pediatric_auprc:.3f}")
print(f"Pediatric MCC: {pediatric_mcc:.3f}")
print(f"Transfer gap (Δ): {transfer_gap:.3f}")

[23:37:09] DEPRECATION WARNING: please use MorganGenerator
[23:37:09] DEPRECATION WARNING: please use MorganGenerator
[23:37:09] DEPRECATION WARNING: please use MorganGenerator
[23:37:09] DEPRECATION WARNING: please use MorganGenerator
[23:37:09] DEPRECATION WARNING: please use MorganGenerator
[23:37:09] DEPRECATION WARNING: please use MorganGenerator
[23:37:09] DEPRECATION WARNING: please use MorganGenerator
[23:37:09] DEPRECATION WARNING: please use MorganGenerator
[23:37:09] DEPRECATION WARNING: please use MorganGenerator
[23:37:09] DEPRECATION WARNING: please use MorganGenerator
[23:37:09] DEPRECATION WARNING: please use MorganGenerator
[23:37:09] DEPRECATION WARNING: please use MorganGenerator
[23:37:09] DEPRECATION WARNING: please use MorganGenerator
[23:37:09] DEPRECATION WARNING: please use MorganGenerator
[23:37:09] DEPRECATION WARNING: please use MorganGenerator
[23:37:09] DEPRECATION WARNING: please use MorganGenerator
[23:37:09] DEPRECATION WARNING: please use MorganGenerat

Compounds evaluated: 137
Adult AUROC (mean across endpoints): 0.838
Pediatric AUROC: 0.532
Pediatric AUPRC: 0.928
Pediatric MCC: 0.000
Transfer gap (Δ): 0.306


In [ ]:
C

    while skip < max_records:
        params = {'search': query, 'limit': limit, 'skip': skip, 'api_key': API_KEY}
        r = requests.get(base_url, params=params)

        if r.status_code == 429:
            print(f"[skip={skip}] Rate limited, waiting 60s")
            time.sleep(60)
            continue
        if r.status_code != 200:
            print(f"[skip={skip}] Status {r.status_code}: {r.text[:300]}")
            break

        data = r.json().get('results', [])
        print(f"[skip={skip}] got {len(data)} records")
        if not data:
            break
        records.extend(data)
        skip += limit
        time.sleep(0.3)

    return records

faers_raw = fetch_faers_pediatric()
print(f"Pulled {len(faers_raw)} pediatric FAERS reports")

[skip=0] got 1000 records
[skip=1000] got 1000 records
[skip=2000] got 1000 records
[skip=3000] got 1000 records
[skip=4000] got 1000 records
[skip=5000] got 1000 records
[skip=6000] got 1000 records
[skip=7000] got 1000 records
[skip=8000] got 1000 records
[skip=9000] got 1000 records
[skip=10000] got 1000 records
[skip=11000] got 1000 records
[skip=12000] got 1000 records
[skip=13000] got 1000 records
[skip=14000] got 1000 records
[skip=15000] got 1000 records
[skip=16000] got 1000 records
[skip=17000] got 1000 records
[skip=18000] got 1000 records
[skip=19000] got 1000 records
[skip=20000] got 1000 records
[skip=21000] got 1000 records
[skip=22000] got 1000 records
[skip=23000] got 1000 records
[skip=24000] got 1000 records
Pulled 25000 pediatric FAERS reports


In [ ]:
faers_df = parse_faers_records(faers_raw)
print(f"{len(faers_df)} drug mentions across {faers_df['drug_name'].nunique()} unique names")

raw_agg = faers_df.groupby('drug_name')['serious'].agg(['mean', 'count']).reset_index()
pediatric_drugs = raw_agg[raw_agg['count'] >= 2].copy()
pediatric_drugs['pediatric_toxic'] = (pediatric_drugs['mean'] >= 0.5).astype(int)
print(f"{len(pediatric_drugs)} drugs with enough reports")

77364 drug mentions across 9179 unique names
4271 drugs with enough reports


In [ ]:
from sklearn.ensemble import RandomForestClassifier
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

def train_and_eval_xgb(X_tr, y_tr, w_tr, X_te, y_te, w_te, tasks):
    trained, res = {}, {}
    for i, task in enumerate(tasks):
        mask = w_tr[:, i] > 0
        m = xgb.XGBClassifier(n_estimators=200, max_depth=6, eval_metric='logloss')
        m.fit(X_tr[mask], y_tr[mask, i])
        trained[task] = m
        test_mask = w_te[:, i] > 0
        preds = m.predict_proba(X_te[test_mask])[:, 1]
        y_true = y_te[test_mask, i]
        res[task] = {'AUROC': roc_auc_score(y_true, preds),
                      'AUPRC': average_precision_score(y_true, preds),
                      'MCC': matthews_corrcoef(y_true, preds > 0.5)}
    return trained, res

def train_and_eval_rf(X_tr, y_tr, w_tr, X_te, y_te, w_te, tasks):
    trained, res = {}, {}
    for i, task in enumerate(tasks):
        mask = w_tr[:, i] > 0
        m = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)
        m.fit(X_tr[mask], y_tr[mask, i])
        trained[task] = m
        test_mask = w_te[:, i] > 0
        preds = m.predict_proba(X_te[test_mask])[:, 1]
        y_true = y_te[test_mask, i]
        res[task] = {'AUROC': roc_auc_score(y_true, preds),
                      'AUPRC': average_precision_score(y_true, preds),
                      'MCC': matthews_corrcoef(y_true, preds > 0.5)}
    return trained, res

def build_dnn(input_dim):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy')
    return model

def train_and_eval_dnn(X_tr, y_tr, w_tr, X_te, y_te, w_te, tasks, input_dim):
    trained, res = {}, {}
    for i, task in enumerate(tasks):
        mask = w_tr[:, i] > 0
        m = build_dnn(input_dim)
        es = callbacks.EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)
        m.fit(X_tr[mask], y_tr[mask, i], epochs=50, batch_size=64, verbose=0, callbacks=[es])
        trained[task] = m
        test_mask = w_te[:, i] > 0
        preds = m.predict(X_te[test_mask], verbose=0).flatten()
        y_true = y_te[test_mask, i]
        res[task] = {'AUROC': roc_auc_score(y_true, preds),
                      'AUPRC': average_precision_score(y_true, preds),
                      'MCC': matthews_corrcoef(y_true, preds > 0.5)}
    return trained, res

xgb_models, xgb_results = train_and_eval_xgb(X_train, y_train, w_train, X_test, y_test, w_test, tasks)
rf_models, rf_results = train_and_eval_rf(X_train, y_train, w_train, X_test, y_test, w_test, tasks)
dnn_models, dnn_results = train_and_eval_dnn(X_train, y_train, w_train, X_test, y_test, w_test, tasks, X_train.shape[1])

def evaluate_pediatric(model_dict, X_ped, y_ped, tasks, is_keras=False):
    if is_keras:
        preds = np.column_stack([model_dict[t].predict(X_ped, verbose=0).flatten() for t in tasks])
    else:
        preds = np.column_stack([model_dict[t].predict_proba(X_ped)[:, 1] for t in tasks])
    risk_score = preds.mean(axis=1)
    return {'AUROC': roc_auc_score(y_ped, risk_score),
            'AUPRC': average_precision_score(y_ped, risk_score),
            'MCC': matthews_corrcoef(y_ped, risk_score > 0.5)}

overlap_mols = [Chem.MolFromSmiles(s) for s in overlap['canonical_smiles']]
X_pediatric = np.array([featurize_mol(m) for m in overlap_mols])
y_pediatric = overlap['pediatric_toxic'].values

summary = []
for name, mdict, res, is_keras in [
    ('XGBoost', xgb_models, xgb_results, False),
    ('Random Forest', rf_models, rf_results, False),
    ('DNN', dnn_models, dnn_results, True),
]:
    adult_mean = np.mean([res[t]['AUROC'] for t in tasks])
    ped = evaluate_pediatric(mdict, X_pediatric, y_pediatric, tasks, is_keras=is_keras)
    summary.append({
        'model': name, 'n_compounds': len(y_pediatric),
        'adult_auroc': adult_mean, 'pediatric_auroc': ped['AUROC'],
        'pediatric_auprc': ped['AUPRC'], 'pediatric_mcc': ped['MCC'],
        'transfer_gap': adult_mean - ped['AUROC']
    })

summary_df = pd.DataFrame(summary)
print(summary_df)

[00:24:01] DEPRECATION WARNING: please use MorganGenerator
[00:24:01] DEPRECATION WARNING: please use MorganGenerator
[00:24:01] DEPRECATION WARNING: please use MorganGenerator
[00:24:01] DEPRECATION WARNING: please use MorganGenerator
[00:24:01] DEPRECATION WARNING: please use MorganGenerator
[00:24:01] DEPRECATION WARNING: please use MorganGenerator
[00:24:01] DEPRECATION WARNING: please use MorganGenerator
[00:24:01] DEPRECATION WARNING: please use MorganGenerator
[00:24:01] DEPRECATION WARNING: please use MorganGenerator
[00:24:01] DEPRECATION WARNING: please use MorganGenerator
[00:24:01] DEPRECATION WARNING: please use MorganGenerator
[00:24:01] DEPRECATION WARNING: please use MorganGenerator
[00:24:01] DEPRECATION WARNING: please use MorganGenerator
[00:24:01] DEPRECATION WARNING: please use MorganGenerator
[00:24:01] DEPRECATION WARNING: please use MorganGenerator
[00:24:01] DEPRECATION WARNING: please use MorganGenerator
[00:24:01] DEPRECATION WARNING: please use MorganGenerat

           model  n_compounds  adult_auroc  pediatric_auroc  pediatric_auprc  \
0        XGBoost          137     0.837843         0.532283         0.927876   
1  Random Forest          137     0.843190         0.439370         0.908877   
2            DNN          137     0.773179         0.536614         0.932366   

   pediatric_mcc  transfer_gap  
0            0.0      0.305560  
1            0.0      0.403820  
2            0.0      0.236565  


In [ ]:
!pip install -q rdkit xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 15.3 MB/s eta 0:00:00


In [ ]:
import os
print(os.getcwd())
print([f for f in os.listdir() if f.endswith(".py")])

/content
['load_tox21_adult[1].py', 'stats_utils[1].py', 'load_faers_pediatric[1].py', 'run_real_pipeline[1].py', 'pediatric_baseline_rigor[1].py', 'adult_baseline_rigor[1].py']


In [ ]:
from run_real_pipeline import main
main(quick_mode=True)

######################################################################
PART 1 / ADULT -- downloading + featurizing real Tox21
######################################################################
unparseable SMILES dropped: 8
duplicate structures collapsed: 234
compounds after dedup: 7589
dropped failed round-trip: 2
final compounds: 7587
descriptor matrix: (7587, 214), any inf: False
scaffolds: 2267   fold sizes: [1698, 1473, 1472, 1472, 1472]

######################################################################
PART 2 / ADULT -- Next-Steps #1-3, RF + XGBoost + DNN
######################################################################

=== ADULT: RF / fp+desc (5-fold scaffold CV, pooled) ===
     task  roc_auc   pr_auc  n_pos
    NR-AR 0.776529 0.533582    299
NR-AR-LBD 0.857334 0.569164    230
pooled mean ROC-AUC 0.817   pooled mean PR-AUC 0.551

--- ADULT: RF / fp+desc -- per-fold mean +/- std (n=5 folds) ---
     task  n_units  roc_auc_mean  roc_auc_std  pr_auc_mean  pr_auc_std


In [ ]:
main(quick_mode=False)